In [1]:
import sys
import traceback
import logging
import portalocker
import os
import re

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

# 경로 설정 - 플랫폼에 따라 다르게 처리
if os.name == 'nt':  # Windows
    sys.path.insert(0, r"Y:/git/pyaedt_library/src/")
else:  # Linux/Unix
    # Linux 서버 경로들 시도
    possible_paths = [
        # r"/gpfs/home1/r1jae262/jupyter/git/pyaedt_library/src/",
        r"../pyaedt_library/src/",
        os.path.abspath(os.path.join(BASE_DIR, "../git/pyaedt_library/src/")),
        "/home1/r1jae262/jupyter/git/pyaedt_library/src/",
        "/home1/dhj02/NEC/git/pyaedt_library/src/",
        "/home1/dw16/NEC/git/pyaedt_library/src/",
        "/home1/harry261/NEC/git/pyaedt_library/src/",
        "/home1/hmlee31/NEC/git/pyaedt_library/src/",
        "/home1/jji0930/NEC/git/pyaedt_library/src/",
        "/home1/wjddn5916/NEC/git/pyaedt_library/src/"
    ]
    for path in possible_paths:
        if os.path.exists(path):
            sys.path.insert(0, path)
            break

import pyaedt_module
from pyaedt_module.core import pyDesktop
import os
import time
from datetime import datetime

import math
import copy

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


import platform
import csv



from module.input_parameter import create_input_parameter, set_design_variables, validation_check
from module.modeling import create_core, create_coil, create_coil_section


from ansys.aedt.core import settings

settings.skip_license_check = True
settings.wait_for_license = False



if os.name == 'nt':  # Windows
    GUI = False
else:  # Linux/Unix
    GUI = True

from filelock import FileLock
import shutil



class Simulation() :

    def __init__(self, desktop=None) :

        self.NUM_CORE = 4
        self.NUM_TASK = 1
        self.desktop = desktop


    def create_simulation_name(self):


        file_path = "./simulation_num.txt"
        simulation_dir = "./simulation"
        os.makedirs(simulation_dir, exist_ok=True)

        # 파일 생성 및 잠금
        with open(file_path, "a+", encoding="utf-8") as file:
            portalocker.lock(file, portalocker.LOCK_EX)
            file.seek(0)
            raw = file.read().strip()

            # simulation 넘버 결정
            if raw.isdigit():
                current_num = int(raw)
            else:
                # 파일에 값이 없거나 손상, simulation 폴더 기준으로 찾기
                current_num = 1
                try:
                    existing_nums = []
                    for name in os.listdir(simulation_dir):
                        m = re.match(r"^simulation(\d+)$", name)
                        if m:
                            existing_nums.append(int(m.group(1)))
                    if existing_nums:
                        current_num = max(existing_nums) + 1
                except Exception:
                    pass

            self.num = current_num
            self.PROJECT_NAME = f"simulation{current_num}"
            next_num = current_num + 1

            # 파일에 다음 넘버 저장
            file.seek(0)
            file.truncate()
            file.write(str(next_num))
            file.flush()

    def create_project(self) :

        # simulation 디렉토리 생성 (존재하지 않으면)
        simulation_dir = "./simulation"
        if not os.path.exists(simulation_dir):
            os.makedirs(simulation_dir, exist_ok=True)
        
        # 절대 경로로 변환
        project_path = os.path.abspath(os.path.join(simulation_dir, self.PROJECT_NAME))
        
        # desktop이 None이거나 유효하지 않은지 확인
        if self.desktop is None:
            raise RuntimeError("Desktop instance is None. Cannot create project.")
        
        try:
            self.project = self.desktop.create_project(path=project_path, name=self.PROJECT_NAME)
        except Exception as e:
            error_msg = f"Failed to create project '{self.PROJECT_NAME}' at path '{project_path}': {e}\n"
            print(error_msg, file=sys.stderr)
            sys.stderr.flush()
            raise

    def create_design(self) :
        self.design1 = self.project.create_design(name="maxwell_design", solver="maxwell3d", solution="AC Magnetic")

        # skip mesh setting
        oDesign = self.design1.odesign
        oDesign.SetDesignSettings(
            [
                "NAME:Design Settings Data",
                "Allow Material Override:=", False,
                "Perform Minimal validation:=", False,
                "EnabledObjects:="	, [],
                "PerfectConductorThreshold:=", 1E+30,
                "InsulatorThreshold:="	, 1,
                "SolveFraction:="	, False,
                "Multiplier:="		, "1",
                "SkipMeshChecks:="	, True
            ], 
            [
                "NAME:Model Validation Settings",
                "EntityCheckLevel:="	, "Strict",
                "IgnoreUnclassifiedObjects:=", False,
                "SkipIntersectionChecks:=", False
            ])

    def create_core(self) :
        self.design1.set_power_ferrite(cm=1.38, x=1.51, y=1.74) # 1K101 parameter [W/m^3]
        self.power_ferrite_mat = self.design1.materials["power_ferrite"]
        self.power_ferrite_mat.permeability = "3000"
        self.design1.main_core = create_core(design=self.design1, name="core", core_material="power_ferrite")

    def create_coil(self) :
        # New design starts without excitations. Windings are created later in assign_winding().

        self.Tx_windings, self.Tx_N, self.Tx_coil_width, self.Tx_coil_height, self.Tx_coil_gap_x, self.Tx_coil_gap_z = create_coil(
            design = self.design1,
            name = "Tx",
            window_height = self.df_plus["nwh1"].iloc[0],
            window_length = self.df_plus["nwl1"].iloc[0],
            window_layer = self.df_plus["N1"].iloc[0],
            N_input = 1,
            width_fill_factor = self.df_plus["wff1"].iloc[0],
            space_length = self.df_plus["sl1"].iloc[0],
            space_width = self.df_plus["sw1"].iloc[0],
            shape = "rectangle",
            offset = [0,0,0],
            color = [255, 10, 10]
        )   

        l1 = self.df_plus["l1"].iloc[0]
        l2 = self.df_plus["l2"].iloc[0]

        self.Rx_windings1, self.Rx_N1, self.Rx_coil_width1, self.Rx_coil_height1, self.Rx_coil_gap_x1, self.Rx_coil_gap_z1 = create_coil(
            design = self.design1,
            name = "Rx_center",
            window_height = self.df_plus["nwh2"].iloc[0],
            window_length = self.df_plus["nwl2_main"].iloc[0],
            window_layer = self.df_plus["N2_main"].iloc[0],
            N_input = 1,
            width_fill_factor = self.df_plus["wff2"].iloc[0],
            space_length = self.df_plus["sl2_main"].iloc[0],
            space_width = self.df_plus["sw2_main"].iloc[0],
            shape = "rectangle",
            offset = [0,0,0],
            color = [10, 10, 255]
        )

        if self.df_plus["N2_side"].iloc[0] > 0 :

            self.Rx_windings2, self.Rx_N2, self.Rx_coil_width2, self.Rx_coil_height2, self.Rx_coil_gap_x2, self.Rx_coil_gap_z2 = create_coil(
                design = self.design1,
                name = "Rx_side1",
                window_height = self.df_plus["nwh2"].iloc[0],
                window_length = self.df_plus["nwl2_side"].iloc[0],
                window_layer = self.df_plus["N2_side"].iloc[0],
                N_input = 1,
                width_fill_factor = self.df_plus["wff2"].iloc[0],
                space_length = self.df_plus["sl2_side"].iloc[0],
                space_width = self.df_plus["sw2_side"].iloc[0],
                shape = "rectangle",
                offset = [(-l1-l2-l1/2),0,0],
                color = [10, 10, 255]
            )       
            self.Rx_windings3, self.Rx_N3, self.Rx_coil_width3, self.Rx_coil_height3, self.Rx_coil_gap_x3, self.Rx_coil_gap_z3 = create_coil(
                design = self.design1,
                name = "Rx_side2",
                window_height = self.df_plus["nwh2"].iloc[0],
                window_length = self.df_plus["nwl2_side"].iloc[0],
                window_layer = self.df_plus["N2_side"].iloc[0], 
                N_input = 1,
                width_fill_factor = self.df_plus["wff2"].iloc[0],
                space_length = self.df_plus["sl2_side"].iloc[0],
                space_width = self.df_plus["sw2_side"].iloc[0],
                shape = "rectangle",
                offset = [(l1+l2+l1/2),0,0],
                color = [10, 10, 255]
            )

        nwl1 = float(self.df_plus["nwl1"])
        nwl2_main = float(self.df_plus["nwl2_main"])
        nwl2_side = float(self.df_plus["nwl2_side"])
        nwh1 = float(self.df_plus["nwh1"])
        nwh2 = float(self.df_plus["nwh2"])

        sl1 = float(self.df_plus["sl1"])
        sw1 = float(self.df_plus["sw1"])
        sl2_main = float(self.df_plus["sl2_main"])
        sw2_main = float(self.df_plus["sw2_main"])
        sl2_side = float(self.df_plus["sl2_side"])
        sw2_side = float(self.df_plus["sw2_side"])

        box_center = self.design1.modeler.create_box(origin=[-(sl2_main+2*nwl2_main)/2, -(sw2_main+2*nwl2_main)/2, -(nwh2)/2], sizes=[sl2_main+2*nwl2_main, sw2_main+2*nwl2_main, nwh2], name="box_center")
        box_center_sub = self.design1.modeler.create_box(origin=[-(sl2_main)/2, -(sw2_main)/2, -(nwh2)/2], sizes=[sl2_main, sw2_main, nwh2], name="box_center_sub")
        self.design1.modeler.subtract(blank_list=[box_center], tool_list=[box_center_sub], keep_originals=False)

        box_side1 = self.design1.modeler.create_box(origin=[-(sl2_side+2*nwl2_side)/2-(l1+l2+l1/2), -(sw2_side+2*nwl2_side)/2, -(nwh2)/2], sizes=[sl2_side+2*nwl2_side, sw2_side+2*nwl2_side, nwh2], name="box_side1")
        box_side1_sub = self.design1.modeler.create_box(origin=[-(sl2_side)/2-(l1+l2+l1/2), -(sw2_side)/2, -(nwh2)/2], sizes=[sl2_side, sw2_side, nwh2], name="box_side1_sub")
        self.design1.modeler.subtract(blank_list=[box_side1], tool_list=[box_side1_sub], keep_originals=False)

        box_side2 = self.design1.modeler.create_box(origin=[-(sl2_side+2*nwl2_side)/2+(l1+l2+l1/2), -(sw2_side+2*nwl2_side)/2, -(nwh2)/2], sizes=[sl2_side+2*nwl2_side, sw2_side+2*nwl2_side, nwh2], name="box_side2")
        box_side2_sub = self.design1.modeler.create_box(origin=[-(sl2_side)/2+(l1+l2+l1/2), -(sw2_side)/2, -(nwh2)/2], sizes=[sl2_side, sw2_side, nwh2], name="box_side2_sub")
        self.design1.modeler.subtract(blank_list=[box_side2], tool_list=[box_side2_sub], keep_originals=False)

        self.design1.modeler.subtract(blank_list=[box_center], tool_list=self.Rx_windings1, keep_originals=True)
        self.design1.modeler.subtract(blank_list=[box_side1], tool_list=self.Rx_windings2, keep_originals=True)
        self.design1.modeler.subtract(blank_list=[box_side2], tool_list=self.Rx_windings3, keep_originals=True)

    def create_coil_section(self) :

        self.Tx_neg_sheets, self.Tx_pos_sheets = create_coil_section(design=self.design1, winding_obj=self.Tx_windings, sheet_prefix = None, plane = "ZX", rename_faces = False)
        self.Rx_neg_sheets_center, self.Rx_pos_sheets_center = create_coil_section(design=self.design1, winding_obj=self.Rx_windings1, sheet_prefix = None, plane = "ZX", rename_faces = False)
        if self.df_plus["N2_side"].iloc[0] != 0 :
            self.Rx_neg_sheets_side1, self.Rx_pos_sheets_side1 = create_coil_section(design=self.design1, winding_obj=self.Rx_windings2, sheet_prefix = None, plane = "ZX", rename_faces = False)
            self.Rx_neg_sheets_side2, self.Rx_pos_sheets_side2 = create_coil_section(design=self.design1, winding_obj=self.Rx_windings3, sheet_prefix = None, plane = "ZX", rename_faces = False)
    
    def assign_winding(self) :

        self.tx_winding = self.design1.assign_winding(
            assignment=[], 
            winding_type="Current", 
            is_solid=True, 
            current=f"{1000*math.sqrt(2)}A",
            name="Tx_winding"
        )

        self.rx_winding = self.design1.assign_winding(
            assignment=[], 
            winding_type="Current", 
            is_solid=True, 
            current=f"{100*math.sqrt(2)}A",
            name="Rx_winding"
        )

    def assign_coil(self) :

        self.Tx_coil = []
        self.Rx_coil = []

        for idx, sheet in enumerate(self.Tx_neg_sheets, start=1):
            coil = self.design1.assign_coil(sheet, conductors_number=1, polarity="Positive", name=f"Tx_coil{idx}")
            self.Tx_coil.append(coil)

        for idx, sheet in enumerate(self.Rx_neg_sheets_center, start=1):
            coil = self.design1.assign_coil(sheet, conductors_number=1, polarity="Negative", name=f"Rx_center_coil{idx}")
            self.Rx_coil.append(coil)

        if self.df_plus["N2_side"].iloc[0] != 0 :
            for idx, sheet in enumerate(self.Rx_neg_sheets_side1 + self.Rx_neg_sheets_side2, start=1):
                coil = self.design1.assign_coil(sheet, conductors_number=1, polarity="Positive", name=f"Rx_side_coil{idx}")
                self.Rx_coil.append(coil)

        self.design1.add_winding_coils(assignment="Tx_winding", coils=[coil.name for coil in self.Tx_coil])
        self.design1.add_winding_coils(assignment="Rx_winding", coils=[coil.name for coil in self.Rx_coil])

        self.design1.assign_matrix(matrix_name="Matrix", assignment=["Tx_winding", "Rx_winding"])

    def assign_skin_depth(self) :

        freq = 1e+3

        mu0 = 4 * math.pi * 1e-7
        mu_copper = mu0 
        sigma_copper = 58000000
        omega = 2 * math.pi * freq
        skin_depth = math.sqrt(2 / (omega * mu_copper * sigma_copper)) * 1e3 # in mm

        self.Tx_skin_depth_mesh = self.design1.mesh.assign_skin_depth(
            assignment=self.Tx_windings,
            skin_depth=f'{skin_depth}mm',
            triangulation_max_length='50mm',
            layers_number="2",
            name="Tx_winding_skin_depth"
        )

    def assign_radiation(self) :

        self.air_region = self.design1.modeler.create_air_region(x_pos=100.0, y_pos=100.0, z_pos=100.0, x_neg=100.0, y_neg=100.0, z_neg=100.0, is_percentage=True)
        self.design1.assign_radiation(assignment=[self.air_region.name], radiation="Radiation")

    def create_setup(self) :

        self.design1.setup = self.design1.create_setup(name = "Setup1")
        self.design1.setup.properties["Max. Number of Passes"] = 8 # 10
        self.design1.setup.properties["Min. Number of Passes"] = 1
        self.design1.setup.properties["Min. Converged Passes"] = 2
        self.design1.setup.properties["Percent Error"] = 2.5 # 2.5
        self.design1.setup.properties["Frequency Setup"] = f"1kHz"

    def get_magnetic_parameter(self) :
        params = [
            ["Matrix.L(Tx_winding,Tx_winding)", f"Ltx", "uH"],
            ["Matrix.L(Rx_winding,Rx_winding)", f"Lrx", "uH"],
            ["Matrix.L(Tx_winding,Rx_winding)", f"M", "uH"],
            ["abs(Matrix.CplCoef(Tx_winding,Rx_winding))", f"k", ""],
            ["Matrix.L(Tx_winding,Tx_winding)*(abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Lmt", "uH"],
            ["Matrix.L(Rx_winding,Rx_winding)*(abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Lmr", "uH"],
            ["Matrix.L(Tx_winding,Tx_winding)*(1-abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Llt", "uH"],
            ["Matrix.L(Rx_winding,Rx_winding)*(1-abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Llr", "uH"],
            ["PerWindingSolidLoss(Tx_winding)", f"Tx_loss", "W"],
            ["PerWindingSolidLoss(Rx_winding)", f"Rx_loss", "W"],
        ]

        dir = self.project.path
        mod = "write"
        import_report = None
        report_name = "magnetic_report"
        file_name = "magnetic_report"

        self.report1, self.df1 = self.design1.get_magnetic_parameter(dir=dir, parameters=params, mod=mod, import_report=import_report, report_name=report_name, file_name=file_name)

        return self.df1

    def save_results_to_csv(self, results_df, filename="simulation_results.csv"):
        """Saves the DataFrame to a CSV file in a process-safe way."""
        lock_path = filename + ".lock"
        with FileLock(lock_path):
            file_exists = os.path.isfile(filename)
            results_df.to_csv(filename, mode='a', header=not file_exists, index=False)
        logging.info(f"Results saved to {filename}")


    def close_project(self):
        self.design1.cleanup_solution()
        self.design1.close_project()
        self.desktop.release_desktop(close_projects=True, close_on_exit=True)

    def delete_project_folder(self):
        time.sleep(10)
        try:
            project_folder = os.path.join(os.getcwd(), "simulation", self.PROJECT_NAME)
            if os.path.isdir(project_folder):
                shutil.rmtree(project_folder)
                logging.info(f"Successfully deleted project folder: {project_folder}")
        except Exception as e:
            logging.error(f"Error deleting project folder {project_folder}: {e}")

  

def run_one_loop(param):
    sim = None
    desktop = None
    try:
        # Use existing Desktop when possible and ensure it closes at context exit.
        with pyDesktop(version=None, non_graphical=GUI, close_on_exit=True, new_desktop=False) as desktop:
            sim = Simulation(desktop=desktop)

            sim.create_simulation_name()
            sim.create_project()
            sim.create_design()

            # create input
            while True:
                sim.input_df = create_input_parameter(param)
                result, sim.df_plus = validation_check(sim.input_df)
                if result:
                    break

            set_design_variables(sim.design1, sim.input_df)

            sim.create_core()
            sim.create_coil()
            sim.create_coil_section()
            sim.assign_winding()
            sim.assign_coil()
            sim.assign_skin_depth()
            sim.assign_radiation()
            sim.create_setup()

            sim.design1.setup.analyze(cores=4)

            sim.get_magnetic_parameter()

            result = pd.concat([sim.df_plus, sim.df1], axis=1)

            try:
                sim.save_results_to_csv(result)
            except Exception as e:
                logging.exception(f"Error saving results to CSV: {e}")

            try:
                sim.close_project()
            except Exception as e:
                logging.exception(f"Error closing project: {e}")

            try:
                pass
                # sim.delete_project_folder()
            except Exception as e:
                logging.exception(f"Error deleting project folder: {e}")
    except Exception as e:
        logging.exception(f"run_one_loop failed: {e}")
        if sim is not None:
            try:
                sim.close_project()
                time.sleep(1)
            except Exception:
                pass
            try:
                pass
                # sim.delete_project_folder()
            except Exception:
                pass
    finally:
        # Safety net: force Desktop release even when internal close fails.
        if desktop is not None:
            try:
                desktop.release_desktop(close_projects=True, close_on_exit=True)
                time.sleep(1)
            except Exception:
                pass


# nano
[N1, N2, N2_main, N2_side] = [6, 60, 25, 35]
[w1, l1, l2, h1] = [300, 75, 200, 450]
[w1c_space_x, w1w2_space_x, w2c_space_x] = [20, 20, 20]
[w1c_space_y, w1w2_space_y, w2w2_space_y, w2c_space_y] = [20, 20, 20, 20]
[w1c_space_z, w2c_space_z] = [20, 20]
[window_ratio, wh1, wh2, wff1, wff2] = [0.4, 0.8, 0.8, 0.5, 0.75]

# amorphous
[N1, N2, N2_main, N2_side] = [6, 60, 30, 30]
[w1, l1, l2, h1] = [500, 75, 200, 500]
[w1c_space_x, w1w2_space_x, w2c_space_x] = [20, 20, 20]
[w1c_space_y, w1w2_space_y, w2w2_space_y, w2c_space_y] = [20, 20, 20, 20]
[w1c_space_z, w2c_space_z] = [20, 20]
[window_ratio, wh1, wh2, wff1, wff2] = [0.4, 0.8, 0.8, 0.5, 0.75]


# # Si-steel
# [N1, N2, N2_main, N2_side] = [8, 80, 25, 35]
# [w1, l1, l2, h1] = [750, 90, 250, 500]
# [w1c_space_x, w1w2_space_x, w2c_space_x] = [20, 20, 20]
# [w1c_space_y, w1w2_space_y, w2w2_space_y, w2c_space_y] = [20, 20, 20, 20]
# [w1c_space_z, w2c_space_z] = [20, 20]
# [window_ratio, wh1, wh2, wff1, wff2] = [0.4, 0.8, 0.8, 1, 0.75]

# # Si-steel
# [N1, N2, N2_main, N2_side] = [6, 60, 59, 1]
# [w1, l1, l2, h1] = [300, 75, 200, 450]
# [w1c_space_x, w1w2_space_x, w2c_space_x] = [20, 20, 20]
# [w1c_space_y, w1w2_space_y, w2w2_space_y, w2c_space_y] = [20, 20, 20, 20]
# [w1c_space_z, w2c_space_z] = [20, 20]
# [window_ratio, wh1, wh2, wff1, wff2] = [0.4, 0.8, 0.8, 1, 0.75]

param_list = [[N1, N2, N2_main, N2_side, w1, l1, l2, h1, w1c_space_x, w1w2_space_x, w2c_space_x, w1c_space_y, w1w2_space_y, w2w2_space_y, w2c_space_y, w1c_space_z, w2c_space_z, window_ratio, wh1, wh2, wff1, wff2]]

input_df = create_input_parameter(param_list[0])
result, df_plus = validation_check(input_df)

df_plus

param_list

[[6,
  60,
  30,
  30,
  500,
  75,
  200,
  500,
  20,
  20,
  20,
  20,
  20,
  20,
  20,
  20,
  20,
  0.4,
  0.8,
  0.8,
  0.5,
  0.75]]

In [4]:
a = [1,2,3,4]
b = [1,2,3]

a+b

[1, 2, 3, 4, 1, 2, 3]

In [ ]:
y = 

In [2]:

arr = pd.DataFrame()  # arr를 빈 DataFrame으로 초기화



param_list = [[N1, N2, N2_main, N2_side, w1, l1, l2, h1, w1c_space_x, w1w2_space_x, w2c_space_x, w1c_space_y, w1w2_space_y, w2w2_space_y, w2c_space_y, w1c_space_z, w2c_space_z, window_ratio, wh1, wh2, wff1, wff2]]

# param_list = [[5, 50, 27, 23, 
# 312, 84, 268, 445, 
# 24.1, 48.3, 49.2, 
# 42.6, 15.9, 47.7, 10.7, 39.9, 40.4, 
# 0.35, 0.9, 0.88, 1, 0.43]]

input_df = create_input_parameter(param_list[0])
result, df_plus = validation_check(input_df)
if result == True :
    arr = pd.concat([arr, df_plus], ignore_index=True)  # arr에 계속 pandas DataFrame을 concat



print(result)
arr

param_list

arr

True


,N1,N2,N2_main,N2_side,w1,l1,l2,h1,w1c_space_x,w1w2_space_x,w2c_space_x,w1c_space_y,w1w2_space_y,w2w2_space_y,w2c_space_y,w1c_space_z,w2c_space_z,window_ratio,wh1,wh2,wff1,wff2,nwl_t,nwl1,nwl2,nwl2_main,nwl2_side,nwh1,nwh2,coil_width1,coil_width2,coil_gap_layer,coil_gap_height,fill_factor,sl1,sw1,sl2_main,sw2_main,sl2_side,sw2_side
0,6,60,30,30,500,75,200,500,20,20,20,20,20,20,20,20,20,0.4,0.8,0.8,0.5,0.75,120,48.0,72.0,36.0,36.0,368.0,368.0,24.0,0.9,0.305085,44.8,0.391304,190,540,326.0,676.0,115,540


In [3]:

def main() :

    for param in param_list :
        try:
            run_one_loop(param)
        except Exception as e:
            logging.exception(f"Error running simulation: {e}")
            continue
        
        finally:
            time.sleep(10)

if __name__ == "__main__":
    main()


PyAEDT INFO: Python version 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\Public\Documents\ESTsoft\CreatorTemp\pyaedt_peets_711ae1d5-5e50-4e85-a754-30eac69548b2.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 62458.
PyAEDT INFO: Electronics Desktop started on gRPC port: 62458 after 26.102252960205078 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v252\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.2 version started with process ID 38392.
PyAEDT INFO: Python version 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)].
PyAEDT INF

INFO:Global:Closing the AEDT Project simulation198


PyAEDT INFO: Project simulation198 closed correctly


INFO:Global:Project simulation198 closed correctly


PyAEDT INFO: Desktop has been released and closed.


...

ERROR:Global:                ^^^^^^^^^^^^^


PyAEDT ERROR: 'nonetype' object has no attribute 'getpath' on close_project


ERROR:Global:'nonetype' object has no attribute 'getpath' on close_project


PyAEDT ERROR: **************************************************************


ERROR:Global:**************************************************************
